# Load Model & Images

In [1]:
!git clone https://github.com/eladrich/pixel2style2pixel.git
%cd pixel2style2pixel

fatal: destination path 'pixel2style2pixel' already exists and is not an empty directory.
/kaggle/working/pixel2style2pixel


In [ ]:
!mkdir -p pretrained_models
!gdown --id 1bMTNWkh5LArlaWSc_wa8VKyq2V42T2z0 -O pretrained_models/psp_ffhq_encode.pt

In [3]:
!mkdir -p directions
!wget -q -O directions/age.npy \
  https://github.com/a312863063/generators-with-stylegan2/raw/master/latent_directions/age.npy
!wget -q -O directions/gender.npy \
  https://github.com/a312863063/generators-with-stylegan2/raw/master/latent_directions/gender.npy

In [26]:
import random
import shutil
from pathlib import Path

# Create output directory
output_dir = Path("/kaggle/working/input_images")
output_dir.mkdir(parents=True, exist_ok=True)

# List all images
images = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png"))]

# Randomly sample 10 images
selected_images = random.sample(images, 10)

# Copy them
for img in selected_images:
    src = os.path.join(IMG_DIR, img)
    dst = output_dir / img
    shutil.copy(src, dst)

print("Saved images:")
for img in selected_images:
    print(img)


Saved images:
016499.jpg
147385.jpg
059939.jpg
004169.jpg
186458.jpg
037296.jpg
048190.jpg
015805.jpg
175837.jpg
051964.jpg


In [4]:
import os, glob
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms

from argparse import Namespace
from models.psp import pSp
from utils.common import tensor2im

# ---------- paths ----------
ckpt_path = "pretrained_models/psp_ffhq_encode.pt"
age_path  = "directions/age.npy"
gender_path  = "directions/gender.npy"
in_dir    = "/kaggle/working/input_images"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- load model ----------
ckpt = torch.load(ckpt_path, map_location="cpu")
opts = ckpt["opts"]
opts["checkpoint_path"] = ckpt_path
opts.setdefault("learn_in_w", False)
opts.setdefault("output_size", 1024)

net = pSp(Namespace(**opts)).to(device).eval()
print("Loaded pSp")

# ---------- load age direction ----------
age_dir = np.load(age_path)  # usually (18,512)
age_dir = torch.from_numpy(age_dir).float().to(device)  # (18,512)
age_dir = age_dir.unsqueeze(0)  # (1,18,512)

# ---------- load gender direction ----------
gender_dir = np.load(gender_path)  # usually (18,512)
gender_dir = torch.from_numpy(gender_dir).float().to(device)  # (18,512)
gender_dir = gender_dir.unsqueeze(0)  # (1,18,512)

# ---------- preprocess ----------
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

W0209 23:01:52.586000 55 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0209 23:01:52.586000 55 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
W0209 23:02:24.618000 55 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W0209 23:02:24.618000 55 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


Loading pSp from checkpoint: pretrained_models/psp_ffhq_encode.pt
Loaded pSp


# Aging

In [5]:
import cv2
import numpy as np
from PIL import Image
import os

haar_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_path)

def align_and_center_pil_opencv(pil_img, out_size=256, scale=1.6):
    """
    Detect face, center square crop, resize to out_size.
    Fallback: simple resize if no face detected.
    """
    img = np.array(pil_img)            # RGB
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    h, w = img.shape[:2]

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(faces) == 0:
        # fallback: just resize whole image
        return pil_img.resize((out_size, out_size), Image.BICUBIC)

    # take the largest detected face
    x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])

    cx = x + fw / 2
    cy = y + fh / 2
    size = max(fw, fh) * scale

    x1 = int(max(0, cx - size / 2))
    y1 = int(max(0, cy - size / 2))
    x2 = int(min(w, cx + size / 2))
    y2 = int(min(h, cy + size / 2))

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        return pil_img.resize((out_size, out_size), Image.BICUBIC)

    return Image.fromarray(crop).resize((out_size, out_size), Image.BICUBIC)

In [ ]:
import torch.nn.functional as F

def cosine_sim_wplus_direction(wplus, direction):
    """
    wplus: (1, 18, 512)
    direction: (1, 18, 512) or (18,512)
    returns scalar cosine similarity
    """
    if direction.dim() == 2:
        direction = direction.unsqueeze(0)

    # flatten
    w = wplus.reshape(1, -1)
    d = direction.reshape(1, -1)

    # center w (helps because absolute offsets can dominate)
    w = w - w.mean(dim=1, keepdim=True)

    return F.cosine_similarity(w, d, dim=1).item()
    
@torch.no_grad()
def decode_from_wplus(wplus):
    y, _ = net.decoder([wplus], input_is_latent=True, randomize_noise=False)
    return y

alphas = [0, -5, -10]

img_paths = sorted(glob.glob(os.path.join(in_dir, "*")))
print("Found", len(img_paths), "images")

img_list = []

for p in img_paths:
    name = os.path.splitext(os.path.basename(p))[0]
    
    img = Image.open(p).convert("RGB")
    img = align_and_center_pil_opencv(img, out_size=256, scale=1.6)
    
    x = transform(img).unsqueeze(0).to(device)
    
    # Invert: get reconstruction + W+ latent
    y_hat, wplus = net(x, randomize_noise=False, return_latents=True)

    row = [img]

    if cosine_sim_wplus_direction(wplus, gender_dir) > 0:
        female = -1
    else:
        female = 1
    
    for a in alphas:
        wplus_aged = wplus + a * (age_dir + female * gender_dir)
        y_aged = decode_from_wplus(wplus_aged)

        out = tensor2im(y_aged[0])
        row.append(out)
    img_list.append(row)

print("Done")

Found 10 images
Done


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

n_rows = len(img_list)
n_cols = len(img_list[0])  # original + edits

plt.figure(figsize=(3 * n_cols, 3 * n_rows))

for i in range(n_rows):
    for j in range(n_cols):
        plt.subplot(n_rows, n_cols, i * n_cols + j + 1)

        img = img_list[i][j]

        # make sure it's PIL
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)

        plt.imshow(img)
        plt.axis("off")

        # titles only on first row
        if i == 0:
            if j == 0:
                plt.title("Original")
            else:
                plt.title(f"α = {alphas[j-1]}")

plt.tight_layout()
plt.show()